In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
from groq import Groq
api_key=os.getenv("api_key")
GROQ_MODEL= "llama-3.3-70b-versatile"
client=Groq(api_key=api_key)

import json
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings


# Declaring some important variable 
Embendding_Model_Name='thenlper/gte-large'
Chroma_Path=r"..\Storage\sindhu_db"
Top_k=5
collection="sindh-10k-collection"
embeddings=SentenceTransformerEmbeddings(model_name=Embendding_Model_Name)



In [11]:
vectore_Store=Chroma(
    collection_name=collection,
    persist_directory=Chroma_Path,
    embedding_function=embeddings
)
# declaring the retreiver
retriever=vectore_Store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":Top_k}
)


In [12]:
# function for retreiving the chunks and returing it in the format of list of dictionaries
def retrieve_chunks(user_query,retriever):
    docs=retriever.invoke(user_query)
    retreived=[]
    for i,doc in enumerate(docs):
        retreived.append({
            "Index":i,
            "Text":doc.page_content,
            "Metadata":doc.metadata
        })
    return retreived
user_query="who wrote indus valley civilisation?"
print(retrieve_chunks(user_query,retriever))

[{'Index': 0, 'Text': "1961).\n11.Fairservis,WalterA.,Jr.,TheOrigin,Character,andDeclineofanEarly\nCivilisation(AmericanMuseumNovitates,no.2302,20Oct.1967).\n12.Gordon,D.H.,ThePrehistoricBackgroundofIndianCulture(Bombay,\n1958).\n13.Khan,F.A.,'Excavations atKotDiji',inPakistanArchaeology,no.2\n(DepartmentofArchaeology,Karachi,1965).\n14.Kosambi,D.D.,TheCultureandCivilisationofAncientIndia(London,\n1965).\n15.Mackay,E.J.H.,FurtherExcavationsatMohenjo-daro(Delhi,1938).\n16.Mackay,E.J.H.,Chanhu-daroExcavations1935-36(AmericanOriental\nSociety,NewHaven,Connecticut,1943).\n17.Marshall,J.,Mohenjo-daroandtheIndusCivilisation(London,1931).\n18.Piggott,S.,PrehistoricIndia(PelicanBooks,Harmondsworth, 1950).\n19.Raikes,Robert,Water,WeatherandPrehistory(London,1967).\n20.Vats,M.S.,ExcavationsatHarappa(Delhi,1940).\n21.Wheeler,R.E.M.(Mortimer),'Harappa1946:theDefencesand\nCemeteryR.37',AncientIndia,no.3(Delhi,1947),pp.58-130.\n22.Wheeler,Mortimer,CivilisationsoftheIndusandBeyond(London,1966).\n141"

In [13]:
# Building the system Prompt
SYSTEM_MESSAGE = """
You are an assistant for a financial services firm that answers user queries on annual reports.
User input will contain the context required to answer the question.
The context will begin with the token #context and contains portions of the source document.
The question will begin with the token #question.
Answer ONLY using the provided context.
If the answer is not found in the context, then do not make things up , just refuse to answers that friendly .
""".strip()
# building the context text from the retreived chunks
def build_context_block(retrieve_chunks):
    parts=[]
    for chunk in retrieve_chunks:
        parts.append(chunk["Text"])

    return "\n\n".join(parts)
# building the user query from the by atatching the context 
def build_user_message(user_query,retrieve_chunks):
    context_text=build_context_block(retrieve_chunks)

    return f"#context\n{context_text}\n#question\n{user_query}"

In [14]:
# Generating the answers:
def generate_answer(conversational_history):
    response = client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=conversational_history,
                    temperature=0
                )

    return response.choices[0].message.content.strip()


In [15]:
# will use this function in case want to see the metadata for inquiry
def rag_answer(user_query, retriever):
    """End-to-end: retrieve → build messages → generate → return audit bundle."""
    retrieved = retrieve_chunks(user_query, retriever)
    user_message = build_user_message(user_query, retrieved)
    answer = generate_answer(SYSTEM_MESSAGE, user_message)
    return {"answer": answer, "retrieved_chunks": retrieved, "user_message": user_message}


In [16]:
# --- Configuration ---
HISTORY_FILE = r"..\Storage\conversation\conversation_history.json"
MAX_STEPS = 5
STOP_WORDS = ["exit", "quit", "stop"]

# --- Helper functions for managing conversation history ---
def load_conversation_history():
    """Loads conversation history from a JSON file, or initializes it with the system message."""
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            history = json.load(f)
        # Ensure system message is always at the beginning if not present
        if not history or history[0]['role'] != 'system':
            history.insert(0, {'role': 'system', 'content': SYSTEM_MESSAGE})
        print(f"Loaded conversation history from {HISTORY_FILE}. Current turns: {len(history)-1}")
    else:
        history = [{'role': 'system', 'content': SYSTEM_MESSAGE}]
        print(f"No conversation history found at {HISTORY_FILE}. Starting a new conversation.")
    return history


In [17]:
def save_conversation_history(history):
    """Saves the current conversation history to a JSON file."""
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Conversation history saved to {HISTORY_FILE}.")


In [18]:
# --- Multi-turn RAG interaction loop ---

conversation_history = load_conversation_history()
turn_count = 0

print("\n--- Starting Multi-Turn RAG Chat ---")
print("Type 'exit', 'quit', or 'stop' to end the conversation.")
print(f"Maximum {MAX_STEPS} turns allowed per session (excluding system message and previous turns).")

# Display initial system message if it's a new conversation
if len(conversation_history) == 1 and conversation_history[0]['role'] == 'system':
        print(f"[SYSTEM] - {conversation_history[0]['content'][:100]}...")

while turn_count < MAX_STEPS:
    user_input = input(f"\n[{turn_count + 1}/{MAX_STEPS}] You: ")

    if user_input.lower() in STOP_WORDS:
        print("Exiting chat. Goodbye!")
        break

    # 1. Retrieve relevant documents from Vector DB
    print("--> Retrieving documents for context...")
    context =retrieve_chunks(user_input, retriever)
    user_message_content = build_user_message(user_input, context).format(context=context, question=user_input)

    # 2. Append the user's message (with context) to history
    conversation_history.append({'role': 'user', 'content': user_message_content})

    # 3. Call the LLM with the entire history
    try:
        print("--> Calling LLM...")

        assistant_response = generate_answer(conversation_history)

    except Exception as e:
        assistant_response = f'Sorry, I encountered the following error: \n {e}'
        print(f"LLM Error: {e}")

    # 4. Append the LLM's response to history
    conversation_history.append({'role': 'assistant', 'content': assistant_response})

    # 5. Save the updated history
    save_conversation_history(conversation_history)

    print(f"Assistant: {assistant_response}")
    turn_count += 1

if turn_count == MAX_STEPS:
    print(f"\nMaximum turns ({MAX_STEPS}) reached. Chat session ended.")





Loaded conversation history from ..\Storage\conversation\conversation_history.json. Current turns: 4

--- Starting Multi-Turn RAG Chat ---
Type 'exit', 'quit', or 'stop' to end the conversation.
Maximum 5 turns allowed per session (excluding system message and previous turns).
--> Retrieving documents for context...
--> Calling LLM...
Conversation history saved to ..\Storage\conversation\conversation_history.json.
Assistant: The author estimates the absolute date range for the Indus civilization to be approximately 2500-1700 B.C. The Mesopotamian ruler's reign used as a benchmark for its peak is Sargon of Agade, whose date is now placed about 2350 B.C.
Exiting chat. Goodbye!
